# DataSteward · Evaluación adversarial

Este notebook clona el repositorio, instala dependencias y corre la evaluación completa con un solo comando.
Por defecto usa el **LLM simulado** (determinista, sin llave). Para Gemini, defina `GEMINI_API_KEY` en los secretos de Colab y cambie `LLM = "gemini"`.

Todos los datos son sintéticos.

In [ ]:
import os, sys
if "google.colab" in sys.modules and not os.path.exists("DataStewardAI"):
    !git clone -q https://github.com/AnthonyHuaccachiA/DataStewardAI.git
if os.path.exists("DataStewardAI"):
    %cd DataStewardAI
elif os.path.basename(os.getcwd()) == "evaluacion":
    %cd ..
!pip install -q -r requirements.txt

In [ ]:
LLM = "simulado"   # o "gemini"
if LLM == "gemini":
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
!python -m src.datasteward.preparar
!python evaluacion/correr_evaluacion.py --llm {LLM}

In [ ]:
from pathlib import Path
from IPython.display import Markdown, Image, display
corrida = Path("evaluacion/resultados") / Path("evaluacion/resultados/ULTIMA_CORRIDA.txt").read_text().strip()
display(Markdown(open(corrida / "resumen.md", encoding="utf-8").read()))
display(Image(str(corrida / "tasas_ataque.png")))
display(Image(str(corrida / "falsos_positivos.png")))

In [ ]:
import pandas as pd
det = pd.read_json(corrida / "detalle.jsonl", lines=True)
# Ataques que pasan con todos los controles activos
det[(det.condicion == "todos_los_controles") & (det.exito == True)][["amenaza", "id", "nota", "evidencia"]]